# OCR bằng API CLC của Lab

Gọi API Hán Nôm của Lab (CLC, `kimhannom.fit.hcmus.edu.vn`) cho từng trang PDF. Kết quả ghi ra
`ocr_raw/<tên sách>/clc_shard0.jsonl`, cùng dạng với hai notebook OCR kia, để đưa vào notebook voting.

**Chuẩn bị:**

1. Tài khoản trên trang `kimhannom.fit.hcmus.edu.vn` đã được Lab cấp cho nhóm trưởng: tên đăng nhập là email của nhóm
   trưởng trong danh sách đăng ký nhóm, mật khẩu GV gửi riêng. Thử đăng nhập trên trình duyệt trước cho chắc.
2. Trong notebook Kaggle: *Add-ons*, *Secrets*, thêm hai secret `CLC_USER` là email đăng nhập và `CLC_PASS` là mật khẩu,
   bật cả hai cho notebook này. Không ghi mật khẩu thẳng vào notebook.

Token đăng nhập của trang chỉ dùng được 1 giờ, nên notebook tự đăng nhập lại trước khi token hết hạn.

**Add Input:** dataset chứa PDF của nhóm. **Không cần GPU**, nhưng phải bật *Internet*. *Run All*, xong thì
*Save Version*.

**Sách in dọc và sách in ngang.** Đầu mỗi cuốn, notebook hỏi API 3 trang ở giữa sách để biết sách in dọc hay in
ngang, không cần khai. Nếu notebook nhận sai thì ghi tên PDF vào `SACH_DOC` hoặc `SACH_NGANG`.

* Sách in dọc: gửi cả trang, mỗi trang 2 lượt gọi, khoảng 5 đến 10 giây.
* Sách in ngang chữ nhỏ, nhất là sách in hai cột: gửi cả trang thì API bỏ sót rất nhiều chữ, có trang mất quá nửa.
  Vì vậy notebook cắt trang thành từng ô theo cột và theo đoạn, phóng sao cho mỗi chữ cao khoảng 70 pixel, rồi gửi
  từng ô và ghép lại theo thứ tự đọc. Mỗi trang mất khoảng 15 đến 30 giây và 10 đến 20 lượt gọi. Notebook tự nhận
  sách in một cột hay hai cột và in ra ở dòng đầu mỗi cuốn; sách hai cột mà bị nhận là một cột thì ghi tên PDF vào
  `SACH_HAI_COT`. Muốn tắt hẳn việc cắt ô thì đặt `CAT_O = False`.

**Giới hạn lượt gọi.** Notebook gọi lần lượt và tự giãn nhịp để không vượt `GIOI_HAN`, đặt sẵn theo quota Lab cấp
cho tài khoản nhóm trưởng. Với quota 10.000 lượt mỗi ngày, sách in ngang chạy được khoảng 600 trang mỗi ngày. Hết lượt
thì notebook dừng, phần đã xong vẫn được ghi. Chạy tiếp: *Add Input* Output của phiên trước, các trang đã xong sẽ không
gọi lại.

In [ ]:
LANG_TYPE = 1         # 0 chưa biết, 1 Hán, 2 Nôm
SACH_DOC = []         # mặc định notebook tự nhận từng trang in dọc hay ngang; chỉ ghi tên PDF vào đây
SACH_NGANG = []       # khi muốn ép hướng cho cả cuốn (bỏ được 1 lượt gọi mỗi trang)
SACH_HAI_COT = []     # sách in ngang hai cột mà notebook không tự nhận ra (xem dòng "hai cột / một cột" khi chạy)
EPITAPH = 0           # 1 nếu là văn bia
TAG = None            # model chuyên biệt do Lab cấp riêng, nếu có; mặc định None
LONG_SIDE = 2000      # cạnh dài của ảnh trang gửi lên, tính bằng pixel
CAT_O = True          # sách in ngang chữ nhỏ: cắt trang thành ô rồi mới gửi (API đọc sót nhiều nếu gửi cả trang)
CO_CHU = 70           # chiều cao mỗi chữ trong ô gửi lên, tính bằng pixel; chữ quá nhỏ hay quá to API đều đọc sót
GIOI_HAN = {60: 60, 3600: 3000, 86400: 10000}  # số giây -> số lượt gọi tối đa, theo quota Lab cấp
BUDGET_H = 11         # Kaggle cắt phiên ở 12 giờ

In [ ]:
import base64, glob, io, json, os, re, shutil, sys, time
from collections import deque
from pathlib import Path
import numpy as np
import requests

os.system(f"{sys.executable} -m pip install -q pypdfium2")
import pypdfium2 as pdfium

SITE = "https://kimhannom.fit.hcmus.edu.vn"
BASE = SITE + "/api/web/clc-sinonom/"
# Tài liệu API: bắt buộc có User-Agent và không được bắt đầu bằng "python-requests"
UA = "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0 Safari/537.36"

def secret(ten):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(ten)
    except Exception:
        return os.environ.get(ten, "")
USER, PASS = secret("CLC_USER"), secret("CLC_PASS")
assert USER and PASS, "Chưa có tài khoản: Add-ons > Secrets > thêm CLC_USER và CLC_PASS (xem hướng dẫn ở đầu notebook)"

S = requests.Session()
HEADERS, HET_HAN = {}, 0.0

def dang_nhap():
    """Đăng nhập trang web lấy cookie token (dùng được 1 giờ), ghi vào HEADERS."""
    global HEADERS, HET_HAN
    w = requests.Session(); w.headers["User-Agent"] = UA
    w.post(SITE + "/account/login", data={"UserName": USER, "Password": PASS}, timeout=60, allow_redirects=False)
    tok = w.cookies.get("token")
    assert tok, "Đăng nhập không được: kiểm tra lại CLC_USER, CLC_PASS bằng cách đăng nhập trên trình duyệt"
    try:
        p = tok.split(".")[1]
        HET_HAN = json.loads(base64.urlsafe_b64decode(p + "=" * (-len(p) % 4)))["exp"] - 300  # làm mới trước 5 phút
    except Exception:
        HET_HAN = time.time() + 1800
    HEADERS = {"Authorization": f"Bearer {tok}", "User-Agent": UA}

dang_nhap(); print("Đăng nhập được với tài khoản", USER)
T0 = time.time()
DA_GOI = deque()  # thời điểm các lượt gọi trong phiên này

class HetQuota(Exception):
    pass

def cho_luot():
    """Đợi cho tới khi gọi thêm 1 lượt vẫn không vượt GIOI_HAN; đợi quá thời gian còn lại của phiên thì dừng."""
    while True:
        now = time.time()
        while DA_GOI and now - DA_GOI[0] > max(GIOI_HAN):
            DA_GOI.popleft()
        doi = 0
        for cua_so, toi_da in GIOI_HAN.items():
            trong = [t for t in DA_GOI if now - t < cua_so]
            if len(trong) >= toi_da:
                doi = max(doi, trong[0] + cua_so - now + 1)
        if doi <= 0:
            DA_GOI.append(now); return
        if now + doi - T0 > BUDGET_H * 3600:
            raise HetQuota(f"đã gọi đủ GIOI_HAN, phải đợi {doi / 3600:.1f} giờ nữa")
        if doi > 60:
            print(f"  đợi {doi / 60:.0f} phút cho đủ giới hạn lượt gọi", flush=True)
        time.sleep(doi)

def goi(api, payload=None, files=None, timeout=300, so_lan=8):
    """POST 1 API của CLC, trả về data. Lỗi mạng thì thử lại. Bị chặn vì quota (mã 000403) thì đợi 15 phút, 4 lần."""
    bi_chan = 0
    for lan in range(so_lan):
        if time.time() > HET_HAN:
            dang_nhap()
        cho_luot()
        try:
            r = S.post(BASE + api, json=payload, files=files, headers=HEADERS, timeout=timeout)
            if r.status_code in (401, 403):  # token hết hạn hoặc bị thu hồi -> đăng nhập lại
                print(f"  {api}: HTTP {r.status_code}, đăng nhập lại"); dang_nhap(); continue
            d = r.json()
        except Exception as e:  # mạng chập chờn / server quá tải
            print(f"  {api}: lỗi {e!r}, thử lại"); time.sleep(20 * (lan + 1)); continue
        if d.get("is_success"):
            return d.get("data") or {}
        if str(d.get("code")) == "000403" or r.status_code == 429:
            bi_chan += 1
            if bi_chan > 4 or time.time() + 900 - T0 > BUDGET_H * 3600:
                raise HetQuota(d)
            print(f"  {api}: bị chặn vì giới hạn lượt gọi ({str(d)[:120]}), đợi 15 phút"); time.sleep(900); continue
        print(f"  {api}: {str(d)[:200]}"); time.sleep(10 * (lan + 1))
    raise RuntimeError(f"{api}: lỗi nhiều lần liền")

def dong(raw):
    return [str(x).strip() for x in raw or [] if str(x).strip()]

def anh(pg):
    img = pg.render(scale=LONG_SIDE / max(pg.get_size())).to_pil().convert("RGB")
    buf = io.BytesIO(); img.save(buf, "JPEG", quality=90)
    return img, buf.getvalue()

def upload(jpg):
    return goi("image-upload", files={"image_file": ("trang.jpg", jpg, "image/jpeg")})["file_name"].strip()

def do_huong(doc):
    """Nhận sách in dọc (1) hay ngang (2): hỏi structure-classification trên 3 trang ở giữa sách, lấy đa số.
    Phải biết hướng vì gọi chế độ chữ dọc cho trang in ngang thì server treo tới 504."""
    phieu = []
    for p in sorted({len(doc) // 3, len(doc) // 2, 2 * len(doc) // 3}):
        try:
            c = goi("structure-classification", {"file_name": upload(anh(doc[p])[1]), "lang_type": 0, "ocr_id": -1,
                                                 "reading_direction": 0}, so_lan=3)
            phieu.append(int(c.get("reading_direction") or 1))
        except RuntimeError:
            pass
    return 2 if phieu.count(2) > phieu.count(1) else 1

def ocr_trang(jpg, lang_type, rd):
    """upload -> image-ocr theo hướng rd; lỗi liên tục thì thử hướng còn lại (trang lời tựa, mục lục có thể khác hướng)."""
    fn = upload(jpg)
    for i, h in enumerate((rd, 3 - rd)):
        try:
            d = goi("image-ocr", {"ocr_id": 4 if h == 2 else 1, "lang_type": lang_type, "reading_direction": h,
                                  "epitaph": EPITAPH, "tag": TAG, "file_name": fn}, so_lan=3 if i == 0 else 2)
            return dong(d.get("result_ocr_text")), {"bbox": d.get("result_bbox") or [], "lang_type": lang_type,
                                                    "ocr_id": 4 if h == 2 else 1, "reading_direction": h}
        except RuntimeError:
            if i:
                raise
            print(f"  hướng {'ngang' if h == 2 else 'dọc'} lỗi liên tục, thử hướng còn lại")

# ---------- Cắt ô cho trang in ngang chữ nhỏ ----------
# Đo thử: API đọc tốt khi mỗi chữ cao khoảng 50 đến 95 pixel và ảnh dài không quá khoảng 26 chữ; trang in hai cột thì
# phải tách cột, vì API nối dòng của hai cột làm một. Nên trang ngang được cắt theo khoảng trắng: tách lề trên dưới,
# tách cột, rồi chia mỗi cột thành đoạn ngắn, cắt ở khe giữa hai dòng.

def co_chu(img):
    """Ước lượng chiều cao chữ (pixel) của trang in ngang: trung vị độ dày các dòng đo trên vài dải dọc hẹp."""
    g = np.asarray(img.convert("L"), dtype=np.uint8)
    muc = g < min(160, int(np.percentile(g, 50)) - 40)
    H, W = muc.shape
    day = []
    for f in (0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8):
        dai = muc[:, int(f * W): int((f + 0.05) * W)]
        for a, b in doan(dai.sum(axis=1) > max(1, 0.02 * dai.shape[1])):
            if b - a >= 0.006 * H:
                day.append(b - a)
    return 1.4 * float(np.median(day)) if day else 0.03 * max(H, W)  # 1.4: dòng đo được mỏng hơn chữ

def doan(mask):
    """Các đoạn True liên tiếp: list (đầu, cuối)."""
    d = np.diff(np.concatenate([[False], mask, [False]]).astype(np.int8))
    return list(zip(np.where(d == 1)[0], np.where(d == -1)[0]))

def mat_muc(img, c):
    """Ảnh nhị phân chỗ có mực (ngưỡng Otsu), đã bỏ các đường kẻ dài (khung, gạch chia cột, viền scan)."""
    g = np.asarray(img.convert("L"), dtype=np.uint8)
    hist = np.bincount(g.ravel(), minlength=256).astype(float)
    w = np.cumsum(hist); mu = np.cumsum(hist * np.arange(256))
    with np.errstate(divide="ignore", invalid="ignore"):
        v = (mu[-1] * w / w[-1] - mu) ** 2 / (w * (w[-1] - w))
    nguong = int(np.nanargmax(v)) if np.isfinite(v).any() else 0   # trang trắng trơn: không có mực
    muc = g < nguong
    for a in (muc, muc.T):
        d = np.diff(np.pad(a, ((0, 0), (1, 1))).astype(np.int8), axis=1)
        st, en = np.argwhere(d == 1), np.argwhere(d == -1)
        dai = en[:, 1] - st[:, 1] > 3 * c
        for (r, s), e in zip(st[dai], en[dai][:, 1]):
            a[r, s:e] = False
    return muc

def khe(muc, x0, y0, x1, y1, truc, tol=0.004):
    """Mép trống hai đầu và các khe trống bên trong theo trục x hoặc y. tol: tỉ lệ mực tối đa vẫn coi là trống."""
    sub = muc[y0:y1, x0:x1]
    prof = sub.sum(axis=0 if truc == "x" else 1)
    n = sub.shape[0] if truc == "x" else sub.shape[1]
    r = doan(prof <= max(2, int(tol * n))); L = len(prof)
    lo = r[0][1] if r and r[0][0] == 0 else 0
    hi = r[-1][0] if r and r[-1][1] == L else L
    return lo, hi, [(a, b) for a, b in r if a > 0 and b < L]

def cat_o(img, c, toi_da=36, tol=0.02, ghi=None):
    """Chia trang in ngang (ảnh LONG_SIDE) thành các ô theo thứ tự đọc. c = co_chu(img). Trả list (x0, y0, x1, y1).
    tol: tỉ lệ mực được phép vắt qua khe cột; sách hai cột dùng 0.06 để tách được cả khi tiêu đề vắt ngang.
    ghi: list, nếu có thì ghi vị trí các khe cột lớn (tính theo bề ngang trang) để nhận sách hai cột."""
    c0 = c / 1.4                       # các ngưỡng dưới đây đo theo độ dày dòng
    muc = mat_muc(img, c0); out = []; lim = toi_da * c0

    def cot(X0, Y0, X1, Y1):           # khe chia cột: rộng, cho phép ít mực vắt qua, hai cột rộng xấp xỉ nhau
        e = X1 - X0
        return [g for g in khe(muc, X0, Y0, X1, Y1, "x", tol)[2] if g[1] - g[0] >= 0.7 * c0
                and ((min(g[0], e - g[1]) >= 12 * c0 and min(g[0], e - g[1]) >= 0.5 * max(g[0], e - g[1]))
                     or (min(g[0], e - g[1]) <= 3 * c0 and e >= 36 * c0))]

    def tach(x0, y0, x1, y1, sau=0):
        lx, hx, _ = khe(muc, x0, y0, x1, y1, "x"); ly, hy, _ = khe(muc, x0, y0, x1, y1, "y")
        if hx <= lx or hy <= ly:
            return
        x0, x1, y0, y1 = x0 + lx, x0 + hx, y0 + ly, y0 + hy
        if muc[y0:y1, x0:x1].sum() < 0.5 * c0 * c0:
            return
        gy = khe(muc, x0, y0, x1, y1, "y")[2]
        W, H = x1 - x0, y1 - y0
        if sau > 40:
            out.append((x0, y0, x1, y1)); return
        doc = lambda m: (tach(x0, y0, x1, y0 + m, sau + 1), tach(x0, y0 + m, x1, y1, sau + 1))
        # 1) dải đầu trang, chân trang: khe rộng hơn hẳn khe giữa các dòng, sát mép
        tb = float(np.median([b - a for a, b in gy])) if gy else 0
        le = [g for g in gy if g[1] - g[0] >= max(0.8 * c0, 2 * tb) and min(g[0], H - g[1]) <= 4 * c0 and H > 12 * c0]
        if le:
            a, b = max(le, key=lambda g: g[1] - g[0]); doc((a + b) // 2); return
        # 2) tách cột, trái trước phải sau
        k = cot(x0, y0, x1, y1)
        if k:
            a, b = max(k, key=lambda g: g[1] - g[0]); m = (a + b) // 2
            if ghi is not None and W > 0.5 * muc.shape[1]:
                ghi.append((x0 + m) / muc.shape[1])
            tach(x0, y0, x0 + m, y1, sau + 1); tach(x0 + m, y0, x1, y1, sau + 1); return
        # 3) tiêu đề hay hình vắt ngang che khe cột: chỉ tách riêng phần che
        if H >= 12 * c0:
            ung = sorted(g for g in gy if g[1] - g[0] >= 0.5 * c0)
            for a, b in ung:
                m = (a + b) // 2
                if H - m >= 6 * c0 and cot(x0, y0 + m, x1, y1):
                    doc(m); return
            for a, b in reversed(ung):
                m = (a + b) // 2
                if m >= 6 * c0 and cot(x0, y0, x1, y0 + m):
                    doc(m); return
        # 4) ô đã đủ nhỏ thì giữ, không thì cắt ở khe dòng gần điểm chia đều nhất
        if max(W, H) <= lim or H <= 3 * c0 or H <= lim or not gy:
            out.append((x0, y0, x1, y1)); return
        dich = H / max(2, int(np.ceil(H / lim)))
        a, b = min(gy, key=lambda g: abs((g[0] + g[1]) / 2 - dich)); doc((a + b) // 2)

    tach(0, 0, muc.shape[1], muc.shape[0])
    return [tuple(int(v) for v in o) for o in out]

def hai_cot(doc):
    """Sách in ngang có phải in hai cột không: thử cắt 9 trang mẫu (không gọi API). Sách hai cột thật thì khe cột
    nằm gần giữa trang và ở đúng một chỗ trên mọi trang; sách một cột chỉ có khe giả, mỗi trang một chỗ."""
    vt, day = [], 0
    for p in sorted({int(len(doc) * k / 12) for k in range(2, 11)}):
        img = anh(doc[p])[0]; c = co_chu(img)
        if max(img.size) / c <= 32:
            continue
        day += 1; ghi = []
        cat_o(img, c, tol=0.06, ghi=ghi)
        giua = [x for x in ghi if 0.4 <= x <= 0.6]
        if giua:
            vt.append(min(giua, key=lambda x: abs(x - 0.5)))
    if day < 3 or len(vt) < 0.6 * day:
        return False
    return float(np.median(np.abs(np.array(vt) - np.median(vt)))) <= 0.02

def ocr_o(pg, img, c, lang_type, tol):
    """OCR trang in ngang theo từng ô. Render lại trang sao cho chữ cao khoảng CO_CHU pixel rồi cắt ô từ ảnh đó."""
    f = min(CO_CHU / c, 2.5)
    lon = pg.render(scale=LONG_SIDE * f / max(pg.get_size())).to_pil().convert("RGB")
    lines, bbox, loi = [], [], 0
    o = cat_o(img, c, tol=tol)
    for (x0, y0, x1, y1) in o:
        p = int(0.3 * c)
        x0, y0, x1, y1 = max(0, x0 - p), max(0, y0 - p), min(img.width, x1 + p), min(img.height, y1 + p)
        buf = io.BytesIO()
        lon.crop((round(x0 * f), round(y0 * f), round(x1 * f), round(y1 * f))).save(buf, "JPEG", quality=90)
        try:
            d = goi("image-ocr", {"ocr_id": 4, "lang_type": lang_type, "reading_direction": 2, "epitaph": EPITAPH,
                                  "tag": TAG, "file_name": upload(buf.getvalue())}, so_lan=3)
        except RuntimeError:
            loi += 1; continue
        lines += dong(d.get("result_ocr_text"))
        for b in d.get("result_bbox") or []:   # đổi toạ độ khung chữ về ảnh trang LONG_SIDE
            try:
                bbox.append([[[x0 + q[0] / f, y0 + q[1] / f] for q in b[0]], b[1]])
            except Exception:
                pass
    if o and loi == len(o):
        raise RuntimeError("mọi ô đều lỗi")
    return lines, {"bbox": bbox, "lang_type": lang_type, "ocr_id": 4, "reading_direction": 2, "o": len(o), "o_loi": loi}

WORK = Path("/kaggle/working"); RAW = WORK / "ocr_raw"; RAW.mkdir(parents=True, exist_ok=True)
for f in glob.glob("/kaggle/input/**/ocr_raw/*/clc_shard*.jsonl", recursive=True):  # chạy tiếp phiên trước
    dst = RAW / Path(f).parent.name / Path(f).name
    if not dst.exists():
        dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy(f, dst); print("Nạp lại", f)

pdfs = sorted(glob.glob("/kaggle/input/**/*.pdf", recursive=True))
assert pdfs, "Không thấy PDF trong /kaggle/input -- đã Add Input dataset chứa PDF chưa?"
dung = False
for src in pdfs:
    stem = Path(src).stem
    book = re.sub(r"[^\w\-]+", "_", stem)  # cùng cách đặt tên sách như notebook OCR mẫu
    out = RAW / book / "clc_shard0.jsonl"; out.parent.mkdir(parents=True, exist_ok=True)
    xong = {json.loads(l)["page"] for l in open(out, encoding="utf-8")} if out.exists() else set()
    doc = pdfium.PdfDocument(src)
    lang = LANG_TYPE
    todo = [p for p in range(len(doc)) if p not in xong]
    if not todo:
        print(f"{book}: đã xong {len(doc)} trang"); continue
    try:
        if stem in SACH_NGANG or book in SACH_NGANG:
            huong, cach = 2, "theo SACH_NGANG"
        elif stem in SACH_DOC or book in SACH_DOC:
            huong, cach = 1, "theo SACH_DOC"
        else:
            huong, cach = do_huong(doc), "tự nhận"
    except HetQuota as e:
        print(f"!!! Dừng: {str(e)[:200]}"); break
    tol, cot_txt = 0.02, ""
    if CAT_O and huong == 2:
        hai = stem in SACH_HAI_COT or book in SACH_HAI_COT or hai_cot(doc)
        tol, cot_txt = (0.06, ", hai cột") if hai else (0.02, ", một cột")
    print(f"{book}: {len(doc)} trang, còn {len(todo)} trang | lang_type {lang} | in {'ngang' if huong == 2 else 'dọc'}{cot_txt} ({cach})")
    for p in todo:
        t = time.time()
        img, jpg = anh(doc[p])
        try:
            c = co_chu(img) if CAT_O and huong == 2 else 0
            if c and max(img.size) / c > 32:   # trang dài hơn khoảng 32 chữ: cắt ô
                lines, meta = ocr_o(doc[p], img, c, lang, tol)
            else:
                lines, meta = ocr_trang(jpg, lang, huong)
        except HetQuota as e:
            print(f"!!! Dừng: {str(e)[:200]}\nPhần đã xong vẫn được ghi. Chạy tiếp ở phiên sau bằng cách Add Input Output phiên này.")
            dung = True; break
        except Exception as e:  # 1 trang lỗi không dừng cả lượt
            print(f"  tr.{p + 1}: bỏ qua, lỗi {e!r}"); continue
        rec = {"page": p, "text": "\n".join(lines), "w": img.width, "h": img.height,
               "sec": round(time.time() - t, 1), **meta}
        with open(out, "a", encoding="utf-8") as f:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        o = f", {meta['o']} ô" + (f" ({meta['o_loi']} ô lỗi)" if meta.get("o_loi") else "") if "o" in meta else ""
        print(f"  tr.{p + 1}: {len(rec['text'])} ký tự{o}, {rec['sec']}s", flush=True)
    if dung:
        break
n = sum(1 for f in RAW.glob("*/clc_shard*.jsonl") for _ in open(f, encoding="utf-8"))
print(f"\nĐã có {n} trang OCR bằng CLC trong {RAW}")